<a href="https://colab.research.google.com/github/renziji/ProteinMPNN/blob/main/colab_notebooks/quickdemo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#ProteinMPNN
This notebook is intended as a quick demo, more features to come!

In [1]:
#@title Clone github repo
# Import standard-library modules used for filesystem access, timing, and path management
import json, time, os, sys, glob

# Only clone the repository if it has not already been downloaded
if not os.path.isdir("ProteinMPNN"):
  # Quietly clone the ProteinMPNN repository from GitHub
  os.system("git clone -q https://github.com/renziji/ProteinMPNN.git")
# Add the cloned repository root to the Python import path so its modules are discoverable
sys.path.append('/content/ProteinMPNN')

In [2]:
#@title Setup Model
# Import plotting and filesystem/utility libraries
import matplotlib.pyplot as plt
import shutil
import warnings
# Import numerical computing and PyTorch deep-learning libraries
import numpy as np
import torch
from torch import optim
from torch.utils.data import DataLoader
from torch.utils.data.dataset import random_split, Subset
import copy
import torch.nn as nn
import torch.nn.functional as F
import random
import os.path
# Import typing helper for annotating heterogeneous checkpoint dictionaries
from typing import Any
# Import ProteinMPNN utilities: losses, graph helpers, featurization, and model class
from protein_mpnn_utils import loss_nll, loss_smoothed, gather_edges, gather_nodes, gather_nodes_t, cat_neighbors_nodes, _scores, _S_to_seq, tied_featurize, parse_PDB
from protein_mpnn_utils import StructureDataset, StructureDatasetPDB, ProteinMPNN

# Select the GPU when available, otherwise fall back to CPU
device: torch.device = torch.device("cuda:0" if (torch.cuda.is_available()) else "cpu")
#v_48_010=version with 48 edges 0.10A noise
model_name: str = "v_48_020" #@param ["v_48_002", "v_48_010", "v_48_020", "v_48_030"]


# Standard deviation of Gaussian noise added to backbone atom coordinates during inference
backbone_noise: float = 0.00               # Standard deviation of Gaussian noise to add to backbone atoms

# Path to the directory holding the pretrained (vanilla) model weights
path_to_model_weights: str = '/content/ProteinMPNN/vanilla_model_weights'
# Hidden feature dimension shared across encoder and decoder layers
hidden_dim: int = 128
# Number of transformer layers in both the encoder and decoder
num_layers: int = 3
# Local alias of the weights folder used to build the checkpoint path
model_folder_path: str = path_to_model_weights
# Guarantee the folder path ends with a trailing slash for clean concatenation
if model_folder_path[-1] != '/':
    model_folder_path = model_folder_path + '/'
# Assemble the full checkpoint file path for the selected model variant
checkpoint_path: str = model_folder_path + f'{model_name}.pt'

# Load the checkpoint dictionary from disk, mapping tensors to the chosen device
checkpoint: dict[str, Any] = torch.load(checkpoint_path, map_location=device)
# Report the number of k-nearest-neighbour edges stored in the checkpoint
print('Number of edges:', checkpoint['num_edges'])
# Retrieve the noise level used during training from the checkpoint metadata
noise_level_print: float = checkpoint['noise_level']
# Display the training noise level in Angstroms
print(f'Training noise level: {noise_level_print}A')
# Instantiate the ProteinMPNN model with the configured hyperparameters and checkpoint edge count
model: ProteinMPNN = ProteinMPNN(num_letters=21, node_features=hidden_dim, edge_features=hidden_dim, hidden_dim=hidden_dim, num_encoder_layers=num_layers, num_decoder_layers=num_layers, augment_eps=backbone_noise, k_neighbors=checkpoint['num_edges'])
# Move the model parameters to the selected device (GPU or CPU)
model.to(device)
# Load the pretrained weights from the checkpoint into the model
model.load_state_dict(checkpoint['model_state_dict'])
# Switch the model to evaluation mode (disables dropout and gradient tracking)
model.eval()
# Confirm that the model weights were loaded successfully
print("Model loaded")

Number of edges: 48
Training noise level: 0.2A
Model loaded


In [3]:
#@title Helper functions
# Import typing helper for annotating heterogeneous PDB dictionaries
from typing import Any

def make_tied_positions_for_homomers(pdb_dict_list: list[dict[str, Any]]) -> dict[str, list[dict[str, list[int]]]]:
    """Build tied-positions mapping so homomer chains stay sequence-identical.

    For every parsed PDB structure, each residue position across all chains is
    tied to the same position index on every other chain, forcing the sampler
    to produce identical sequences on symmetric (homomer) chains.

    Args:
        pdb_dict_list: List of parsed PDB dictionaries (one per structure) as
            produced by ``parse_PDB``. Each dict carries ``seq_chain_<X>``
            entries and a ``name`` field.

    Returns:
        Mapping from structure name to a list of tied-position dicts. Each
        tied-position dict maps a chain id (e.g. ``"A"``) to a single-element
        list holding the 1-based residue index that should be tied.

        Example for a homotrimer named '1O91' with chains A, B, C, each
        containing 3 residues:

            {
                '1O91': [
                    {'A': [1], 'B': [1], 'C': [1]},
                    {'A': [2], 'B': [2], 'C': [2]},
                    {'A': [3], 'B': [3], 'C': [3]},
                ]
            }
    """
    # Accumulator mapping each structure name to its list of tied positions
    tied_positions_dict: dict[str, list[dict[str, list[int]]]] = {}
    # Iterate over each parsed PDB structure dictionary
    for pdb_dict in pdb_dict_list:
        # Extract and sort single-character chain ids from the seq_chain_* keys (A, B, C, ...)
        all_chain_id_list: list[str] = sorted([key[-1:] for key in list(pdb_dict) if key[:9]=='seq_chain']) #A, B, C, ...
        # List of per-position tied mappings for the current structure
        tied_positions_list: list[dict[str, list[int]]] = []
        # Length of the first chain (assumes equal-length homomer chains)
        chain_length: int = len(pdb_dict[f"seq_chain_{all_chain_id_list[0]}"])
        # Build a tied mapping for every 1-based residue position
        for i in range(1, chain_length+1):
            # Temporary dict collecting the tied index per chain at this position
            temp_dict: dict[str, list[int]] = {}
            # Assign the current position index to every chain (value must be a list)
            for chain in all_chain_id_list:
                temp_dict[chain] = [i] #needs to be a list
            # Store this position's tied mapping in the structure's list
            tied_positions_list.append(temp_dict)
        # Associate the completed tied-positions list with the structure's name
        tied_positions_dict[pdb_dict['name']] = tied_positions_list
    # Return the complete tied-positions dictionary
    return tied_positions_dict

Examples:

1) pdb: 6MRR, homomer: False, designed_chain: A

2) pdb: 1O91, homomer: True, designed_chain: A B C, for correct symmetric tying lenghts of homomer chains should be the same

In [4]:
# Import regular-expression utilities for splitting chain-id strings
import re
# Import the Colab file-upload widget for loading user-supplied PDB files
from google.colab import files
# Import NumPy for array-based amino-acid mask construction
import numpy as np
# Import typing helper for annotating heterogeneous PDB dictionaries
from typing import Any
# Import the canonical amino-acid alphabet from the central constants module
from constants import PROTEIN_ALPHABET

#########################
def get_pdb(pdb_code: str = "") -> str:
  """Resolve a PDB structure to a local ``.pdb`` file path.

  When ``pdb_code`` is empty the user is prompted to upload a PDB file via
  the Colab widget; otherwise the structure is downloaded from the RCSB
  Protein Data Bank using the supplied 4-character code.

  Args:
      pdb_code: A 4-character RCSB PDB code (e.g. ``"1O91"``). Pass an empty
          string (or ``None``) to trigger an interactive file upload.

  Returns:
      The local file path to the resolved ``.pdb`` file - ``"tmp.pdb"`` for
      uploads or ``"<pdb_code>.pdb"`` for downloads.
  """
  # If no PDB code was supplied, fall back to an interactive upload
  if pdb_code is None or pdb_code == "":
    # Launch the Colab upload widget and capture the returned file mapping
    upload_dict = files.upload()
    # Grab the raw bytes of the first (and only) uploaded file
    pdb_string = upload_dict[list(upload_dict.keys())[0]]
    # Persist the uploaded bytes to a temporary local PDB file
    with open("tmp.pdb","wb") as out: out.write(pdb_string)
    # Return the path to the freshly written temporary file
    return "tmp.pdb"
  else:
    # Quietly download the requested PDB file from the RCSB server
    os.system(f"wget -qnc https://files.rcsb.org/view/{pdb_code}.pdb")
    # Return the path to the downloaded PDB file
    return f"{pdb_code}.pdb"

#@markdown ### Input Options
# RCSB PDB code of the target structure (leave blank to upload a file instead)
pdb: str = '1O91' #@param {type:"string"}
# Resolve the PDB code (or upload) into a local file path
pdb_path: str = get_pdb(pdb)
#@markdown - pdb code (leave blank to get an upload prompt)

# Whether the target is a homomer whose symmetric chains should be tied
homomer: bool = True #@param {type:"boolean"}
# Space-separated chain ids to redesign (e.g. "A B C")
designed_chain: str = "A B C" #@param {type:"string"}
# Space-separated chain ids to keep fixed during design (e.g. "D E")
fixed_chain: str = "" #@param {type:"string"}

# Parse the designed-chain string into a list, handling empty input
if designed_chain == "":
  # No chains to design yields an empty list
  designed_chain_list: list[str] = []
else:
  # Replace non-letter separators with commas and split into chain ids
  designed_chain_list = re.sub("[^A-Za-z]+",",", designed_chain).split(",")

# Parse the fixed-chain string into a list, handling empty input
if fixed_chain == "":
  # No chains to fix yields an empty list
  fixed_chain_list: list[str] = []
else:
  # Replace non-letter separators with commas and split into chain ids
  fixed_chain_list = re.sub("[^A-Za-z]+",",", fixed_chain).split(",")

# Build the unique, de-duplicated set of all chains (designed + fixed)
chain_list: list[str] = list(set(designed_chain_list + fixed_chain_list))

#@markdown - specified which chain(s) to design and which chain(s) to keep fixed.
#@markdown   Use comma:`A,B` to specifiy more than one chain

#chain = "A" #@param {type:"string"}
#pdb_path_chains = chain
##@markdown - Define which chain to redesign

#@markdown ### Design Options
# Number of sequence samples to generate per target backbone
num_seqs: int = 1 #@param ["1", "2", "4", "8", "16", "32", "64"] {type:"raw"}
# Alias of num_seqs used by the downstream sampling loop
num_seq_per_target: int = num_seqs

#@markdown - Sampling temperature for amino acids, T=0.0 means taking argmax, T>>1.0 means sample randomly.
# Sampling temperature(s) controlling amino-acid sampling randomness
sampling_temp: str = "0.1" #@param ["0.0001", "0.1", "0.15", "0.2", "0.25", "0.3", "0.5"]



# Flag: save per-position score (-log probability) to .npy files
save_score: int = 0                      # 0 for False, 1 for True; save score=-log_prob to npy files
# Flag: save MPNN predicted probabilities per position
save_probs: int = 0                      # 0 for False, 1 for True; save MPNN predicted probabilites per position
# Flag: only score the input backbone-sequence pairs (no sampling)
score_only: int = 0                      # 0 for False, 1 for True; score input backbone-sequence pairs
# Flag: output conditional probabilities p(s_i given the rest of the sequence and backbone)
conditional_probs_only: int = 0          # 0 for False, 1 for True; output conditional probabilities p(s_i given the rest of the sequence and backbone)
# Flag: output conditional probabilities p(s_i given backbone only)
conditional_probs_only_backbone: int = 0 # 0 for False, 1 for True; if true output conditional probabilities p(s_i given backbone)

# Number of sequences processed per GPU batch (reduce if out of memory)
batch_size: int = 1                      # Batch size; can set higher for titan, quadro GPUs, reduce this if running out of GPU memory
# Maximum allowed sequence length (residues) per target
max_length: int = 20000                  # Max sequence length

# Directory where output sequences will be written
out_folder: str = '.'                    # Path to a folder to output sequences, e.g. /home/out/
# Optional path to a folder of pre-parsed PDBs in jsonl format
jsonl_path: str = ''                     # Path to a folder with parsed pdb into jsonl
# Amino acids to exclude from generated sequences (e.g. 'AC' omits Ala and Cys)
omit_AAs: str = 'X'                      # Specify which amino acids should be omitted in the generated sequence, e.g. 'AC' would omit alanine and cystine.

# Weight blending PSSM bias with MPNN predictions in [0.0, 1.0]
pssm_multi: float = 0.0                    # A value between [0.0, 1.0], 0.0 means do not use pssm, 1.0 ignore MPNN predictions
# Threshold restricting which AAs are allowed per position when using PSSM
pssm_threshold: float = 0.0                # A value between -inf + inf to restric per position AAs
# Flag: use PSSM log-odds as the bias source
pssm_log_odds_flag: int = 0               # 0 for False, 1 for True
# Flag: apply per-residue PSSM bias during sampling
pssm_bias_flag: int = 0                   # 0 for False, 1 for True


##############################################################

# Output folder derived from the out_folder setting
folder_for_outputs: str = out_folder

# Number of full batches needed to produce the requested number of sequences
NUM_BATCHES: int = num_seq_per_target//batch_size
# Number of independent copies run per batch (mirrors batch_size)
BATCH_COPIES: int = batch_size
# List of float sampling temperatures parsed from the comma-separated string
temperatures: list[float] = [float(item) for item in sampling_temp.split()]
# String of amino acids to omit (kept as a raw string for membership tests)
omit_AAs_list: str = omit_AAs
# Canonical amino-acid alphabet imported from the central constants module
alphabet: str = PROTEIN_ALPHABET

# Boolean mask (float32) marking which alphabet positions are omitted
omit_AAs_np: np.ndarray = np.array([AA in omit_AAs_list for AA in alphabet]).astype(np.float32)

# Optional per-structure chain-id mapping (populated later); None means use defaults
chain_id_dict = None
# Optional per-structure fixed-position mapping; None means no fixed residues
fixed_positions_dict = None
# Optional per-structure PSSM mapping; None means no PSSM bias
pssm_dict = None
# Optional per-structure omit-AA mapping; None means no per-structure omits
omit_AA_dict = None
# Optional per-residue AA bias mapping; None means no bias
bias_AA_dict = None
# Optional per-structure tied-positions mapping; None means no tying
tied_positions_dict = None
# Optional per-residue bias mapping; None means no per-residue bias
bias_by_res_dict = None
# Zero-initialised per-alphabet AA bias vector used during sampling
bias_AAs_np: np.ndarray = np.zeros(len(alphabet))


###############################################################
# Parse the target PDB file into a list of per-structure dictionaries
pdb_dict_list: list[dict[str, Any]] = parse_PDB(pdb_path, input_chain_list=chain_list)
# Wrap the parsed structures in a StructureDatasetPDB for batched iteration
dataset_valid: StructureDatasetPDB = StructureDatasetPDB(pdb_dict_list, truncate=None, max_length=max_length)

# Build the chain-id dict mapping the structure name to (designed_chain_list, fixed_chain_list) lists
chain_id_dict = {}
# Map the first structure's name to its designed and fixed chain lists
chain_id_dict[pdb_dict_list[0]['name']]= (designed_chain_list, fixed_chain_list)

# Print the chain-id mapping for inspection
print(chain_id_dict)
# Report the length of each chain in the parsed structure
for chain in chain_list:
  # Length (residue count) of the current chain
  l: int = len(pdb_dict_list[0][f"seq_chain_{chain}"])
  # Print the chain id and its residue count
  print(f"Length of chain {chain} is {l}")

# If the target is a homomer, build tied positions to enforce symmetry
if homomer:
  # Generate tied-positions dict constraining homomer chains to identical sequences
  tied_positions_dict = make_tied_positions_for_homomers(pdb_dict_list)
  print(f"Tied positions for homomer design: {tied_positions_dict}")
else:
  # No tying for non-homomer targets
  tied_positions_dict = None

discarded: {'bad_chars': 0, 'too_long': 0}
{'1O91': (['A', 'B', 'C'], [])}
Length of chain C is 131
Length of chain A is 131
Length of chain B is 131
Tied positions for homomer design: {'1O91': [{'A': [1], 'B': [1], 'C': [1]}, {'A': [2], 'B': [2], 'C': [2]}, {'A': [3], 'B': [3], 'C': [3]}, {'A': [4], 'B': [4], 'C': [4]}, {'A': [5], 'B': [5], 'C': [5]}, {'A': [6], 'B': [6], 'C': [6]}, {'A': [7], 'B': [7], 'C': [7]}, {'A': [8], 'B': [8], 'C': [8]}, {'A': [9], 'B': [9], 'C': [9]}, {'A': [10], 'B': [10], 'C': [10]}, {'A': [11], 'B': [11], 'C': [11]}, {'A': [12], 'B': [12], 'C': [12]}, {'A': [13], 'B': [13], 'C': [13]}, {'A': [14], 'B': [14], 'C': [14]}, {'A': [15], 'B': [15], 'C': [15]}, {'A': [16], 'B': [16], 'C': [16]}, {'A': [17], 'B': [17], 'C': [17]}, {'A': [18], 'B': [18], 'C': [18]}, {'A': [19], 'B': [19], 'C': [19]}, {'A': [20], 'B': [20], 'C': [20]}, {'A': [21], 'B': [21], 'C': [21]}, {'A': [22], 'B': [22], 'C': [22]}, {'A': [23], 'B': [23], 'C': [23]}, {'A': [24], 'B': [24], 'C':

In [5]:
#@title RUN
with torch.no_grad():
  print('Generating sequences...')
  for ix, protein in enumerate(dataset_valid):
    score_list = []
    all_probs_list = []
    all_log_probs_list = []
    S_sample_list = []
    batch_clones = [copy.deepcopy(protein) for i in range(BATCH_COPIES)]
    X, S, mask, lengths, chain_M, chain_encoding_all, chain_list_list, visible_list_list, masked_list_list, masked_chain_length_list_list, chain_M_pos, omit_AA_mask, residue_idx, dihedral_mask, tied_pos_list_of_lists_list, pssm_coef, pssm_bias, pssm_log_odds_all, bias_by_res_all, tied_beta = tied_featurize(batch_clones, device, chain_id_dict, fixed_positions_dict, omit_AA_dict, tied_positions_dict, pssm_dict, bias_by_res_dict)
    pssm_log_odds_mask = (pssm_log_odds_all > pssm_threshold).float() #1.0 for true, 0.0 for false
    name_ = batch_clones[0]['name']

    randn_1 = torch.randn(chain_M.shape, device=X.device)
    log_probs = model(X, S, mask, chain_M*chain_M_pos, residue_idx, chain_encoding_all, randn_1)
    mask_for_loss = mask*chain_M*chain_M_pos
    scores = _scores(S, log_probs, mask_for_loss)
    native_score = scores.cpu().data.numpy()

    for temp in temperatures:
        for j in range(NUM_BATCHES):
            randn_2 = torch.randn(chain_M.shape, device=X.device)
            if tied_positions_dict == None:
                sample_dict = model.sample(X, randn_2, S, chain_M, chain_encoding_all, residue_idx, mask=mask, temperature=temp, omit_AAs_np=omit_AAs_np, bias_AAs_np=bias_AAs_np, chain_M_pos=chain_M_pos, omit_AA_mask=omit_AA_mask, pssm_coef=pssm_coef, pssm_bias=pssm_bias, pssm_multi=pssm_multi, pssm_log_odds_flag=bool(pssm_log_odds_flag), pssm_log_odds_mask=pssm_log_odds_mask, pssm_bias_flag=bool(pssm_bias_flag), bias_by_res=bias_by_res_all)
                S_sample = sample_dict["S"]
            else:
                sample_dict = model.tied_sample(X, randn_2, S, chain_M, chain_encoding_all, residue_idx, mask=mask, temperature=temp, omit_AAs_np=omit_AAs_np, bias_AAs_np=bias_AAs_np, chain_M_pos=chain_M_pos, omit_AA_mask=omit_AA_mask, pssm_coef=pssm_coef, pssm_bias=pssm_bias, pssm_multi=pssm_multi, pssm_log_odds_flag=bool(pssm_log_odds_flag), pssm_log_odds_mask=pssm_log_odds_mask, pssm_bias_flag=bool(pssm_bias_flag), tied_pos=tied_pos_list_of_lists_list[0], tied_beta=tied_beta, bias_by_res=bias_by_res_all)
            # Compute scores
                S_sample = sample_dict["S"]
            log_probs = model(X, S_sample, mask, chain_M*chain_M_pos, residue_idx, chain_encoding_all, randn_2, use_input_decoding_order=True, decoding_order=sample_dict["decoding_order"])
            mask_for_loss = mask*chain_M*chain_M_pos
            scores = _scores(S_sample, log_probs, mask_for_loss)
            scores = scores.cpu().data.numpy()
            all_probs_list.append(sample_dict["probs"].cpu().data.numpy())
            all_log_probs_list.append(log_probs.cpu().data.numpy())
            S_sample_list.append(S_sample.cpu().data.numpy())
            for b_ix in range(BATCH_COPIES):
                masked_chain_length_list = masked_chain_length_list_list[b_ix]
                masked_list = masked_list_list[b_ix]
                seq_recovery_rate = torch.sum(torch.sum(torch.nn.functional.one_hot(S[b_ix], 21)*torch.nn.functional.one_hot(S_sample[b_ix], 21),axis=-1)*mask_for_loss[b_ix])/torch.sum(mask_for_loss[b_ix])
                seq = _S_to_seq(S_sample[b_ix], chain_M[b_ix])
                score = scores[b_ix]
                score_list.append(score)
                native_seq = _S_to_seq(S[b_ix], chain_M[b_ix])
                if b_ix == 0 and j==0 and temp==temperatures[0]:
                    start = 0
                    end = 0
                    list_of_AAs = []
                    for mask_l in masked_chain_length_list:
                        end += mask_l
                        list_of_AAs.append(native_seq[start:end])
                        start = end
                    native_seq = "".join(list(np.array(list_of_AAs)[np.argsort(masked_list)]))
                    l0 = 0
                    for mc_length in list(np.array(masked_chain_length_list)[np.argsort(masked_list)])[:-1]:
                        l0 += mc_length
                        native_seq = native_seq[:l0] + '/' + native_seq[l0:]
                        l0 += 1
                    sorted_masked_chain_letters = np.argsort(masked_list_list[0])
                    print_masked_chains = [masked_list_list[0][i] for i in sorted_masked_chain_letters]
                    sorted_visible_chain_letters = np.argsort(visible_list_list[0])
                    print_visible_chains = [visible_list_list[0][i] for i in sorted_visible_chain_letters]
                    native_score_print = np.format_float_positional(np.float32(native_score.mean()), unique=False, precision=4)
                    line = '>{}, score={}, fixed_chains={}, designed_chains={}, model_name={}\n{}\n'.format(name_, native_score_print, print_visible_chains, print_masked_chains, model_name, native_seq)
                    print(line.rstrip())
                start = 0
                end = 0
                list_of_AAs = []
                for mask_l in masked_chain_length_list:
                    end += mask_l
                    list_of_AAs.append(seq[start:end])
                    start = end

                seq = "".join(list(np.array(list_of_AAs)[np.argsort(masked_list)]))
                l0 = 0
                for mc_length in list(np.array(masked_chain_length_list)[np.argsort(masked_list)])[:-1]:
                    l0 += mc_length
                    seq = seq[:l0] + '/' + seq[l0:]
                    l0 += 1
                score_print = np.format_float_positional(np.float32(score), unique=False, precision=4)
                seq_rec_print = np.format_float_positional(np.float32(seq_recovery_rate.detach().cpu().numpy()), unique=False, precision=4)
                line = '>T={}, sample={}, score={}, seq_recovery={}\n{}\n'.format(temp,b_ix,score_print,seq_rec_print,seq)
                print(line.rstrip())


all_probs_concat = np.concatenate(all_probs_list)
all_log_probs_concat = np.concatenate(all_log_probs_list)
S_sample_concat = np.concatenate(S_sample_list)

Generating sequences...
>1O91, score=1.3158, fixed_chains=[], designed_chains=['A', 'B', 'C'], model_name=v_48_020
EMPAFTAELTVPFPPVGAPVKFDKLLYNGRQNYNPQTGIFTCEVPGVYYFAYHVHCKGGNVWVALFKNNEPMMYTYDEYKKGFLDQASGSAVLLLRPGDQVFLQMPSEQAAGLYAGQYVHSSFSGYLLYPM/EMPAFTAELTVPFPPVGAPVKFDKLLYNGRQNYNPQTGIFTCEVPGVYYFAYHVHCKGGNVWVALFKNNEPMMYTYDEYKKGFLDQASGSAVLLLRPGDQVFLQMPSEQAAGLYAGQYVHSSFSGYLLYPM/EMPAFTAELTVPFPPVGAPVKFDKLLYNGRQNYNPQTGIFTCEVPGVYYFAYHVHCKGGNVWVALFKNNEPMMYTYDEYKKGFLDQASGSAVLLLRPGDQVFLQMPSEQAAGLYAGQYVHSSFSGYLLYPM
>T=0.1, sample=0, score=0.6544, seq_recovery=0.4809
TVEAFTALLTTPNPAVGEPIKFDKVLYNGGNVYDPATGIFTVKTPGIYLFQFVLYVSGADLHVELMLNDTPILKVRLDTVPGKLNQVSGSAILELKEGDKVYIKIPSASANGLYASADAHSYFSGYLLTPL/TVEAFTALLTTPNPAVGEPIKFDKVLYNGGNVYDPATGIFTVKTPGIYLFQFVLYVSGADLHVELMLNDTPILKVRLDTVPGKLNQVSGSAILELKEGDKVYIKIPSASANGLYASADAHSYFSGYLLTPL/TVEAFTALLTTPNPAVGEPIKFDKVLYNGGNVYDPATGIFTVKTPGIYLFQFVLYVSGADLHVELMLNDTPILKVRLDTVPGKLNQVSGSAILELKEGDKVYIKIPSASANGLYASADAHSYFSGYLLTPL


In [6]:
#@markdown ### Amino acid probabilties
import plotly.express as px
fig = px.imshow(np.exp(all_log_probs_concat).mean(0).T,
                labels=dict(x="positions", y="amino acids", color="probability"),
                y=list(alphabet),
                template="simple_white"
               )

fig.update_xaxes(side="top")


fig.show()

In [7]:
#@markdown ### Sampling temperature adjusted amino acid probabilties
import plotly.express as px
fig = px.imshow(all_probs_concat.mean(0).T,
                labels=dict(x="positions", y="amino acids", color="probability"),
                y=list(alphabet),
                template="simple_white"
               )

fig.update_xaxes(side="top")


fig.show()